In [10]:
from tikzpy import (
    TikzPicture,
    Rectangle,
    Point,
    Line,
    PlotCoordinates,
    Scope,
)

tikz = TikzPicture(center=True)

# Improved dimensions based on reference
ORIGIN = Point(0, 0)
input_pos = ORIGIN

# Better proportioned dimensions
layer_block_size = 0.8      # Larger for better readability
embedding_h = 1.25          # Match reference height
linear_h = 0.5              # Match reference height
layer_block_width = 2.0     # Wider for better text fit

def input_block(position, text=None):
    box = tikz.rectangle_from_center(
        position, width=layer_block_width, height=embedding_h, options="fill=red!10"
    )
    tikz.node(box.center, options="align=center", text="Input \\\\ Embedding")
    label = f'{text}' if text is not None else "Input Token"
    tikz.node(box.south - (0, 0.7), text=label)
    tikz.line(box.south - (0, 0.45), box.south)
    return box

def sigmoid_block(position):
    sigmoid = tikz.rectangle_from_south(
        position+(0, 0.4), width=layer_block_width, height=linear_h, options="fill=green!10"
    )
    tikz.node(sigmoid.center, options="align=center", text="Sigmoid")
    tikz.line(position, sigmoid.south)
    return sigmoid

def group_sum_block(position):
    group_sum = tikz.rectangle_from_south(
        position+(0, 0.4), width=layer_block_width, height=linear_h, options="fill=Yellow!10"
    )
    tikz.node(group_sum.center, options="align=center", text="Group Sum")
    tikz.line(position, group_sum.south)
    return group_sum

def output_probabilities(position, target_token=None):
    prob_box = tikz.rectangle_from_south(
        position+(0, 0.4), width=layer_block_width, height=linear_h, options="fill=Purple!10"
    )
    tikz.node(prob_box.center, options="align=center", text="Softmax")
    
    tikz.line(position, prob_box.south)
    
    if target_token:
        label = f'{target_token}'
        tikz.node(prob_box.north + (0, 0.7), text=label, options="align=center")
        tikz.line(prob_box.north, prob_box.north + (0, 0.5))
        
    return prob_box

def add_symbol(pos, flip=True):
    scope = Scope()
    eps = 0.0
    circle = scope.circle(pos, radius=0.25, options="ultra thick")
    scope.line(circle.north - (0, eps), circle.south + (0, eps), options="very thick")
    scope.line(circle.west + (eps, 0), circle.east - (eps, 0), options="very thick")
    tikz.draw(scope)

    # We'll skip the positional encoding for this network since it's not needed
    return circle

def horizontal_layer_block(position, label=None):
    k_layer = tikz.rectangle_from_center(
        position, width=layer_block_size, height=layer_block_size, options="fill=ProcessBlue!10"
    )
    
    if label:
        tikz.node(k_layer.center, options="align=center", text=f"{label}")
    
    return k_layer

def vertical_layer_block(position, label=None):
    base_position = position + (0, 0.4)
    
    container = tikz.rectangle_from_south(
        base_position, width=layer_block_size, height=layer_block_size, options="fill=ProcessBlue!10"
    )
    
    if label:
        tikz.node(container.center, options="align=center", text=f"{label}")
    
    tikz.line(position, container.south)
    
    return container, container.north

def create_zoom_effect(layer_block, offset_distance=1.0):
    """Create a zoom-in effect for a layer block"""
    # Position for small circle (top-right of the layer block with some distance to edge)
    small_circle_pos = Point(
        layer_block.east.x - 0.15,  # Small distance from right edge
        layer_block.north.y - 0.15   # Small distance from top edge
    )
    
    # Position for large circle (much higher above the layer block)
    large_circle_pos = small_circle_pos + (0, offset_distance + 2.0)
    
    # Create small circle - using tikz.circle directly like in your reference
    small_circle = tikz.circle(small_circle_pos, radius=0.08, options="thin, black")
    
    # Create much larger circle - using tikz.circle directly like in your reference
    large_circle = tikz.circle(large_circle_pos, radius=1.5, options="thin, black, fill=white")
    
    # Connect with dashed lines - MARKED AS DASHED SO STYLING LOOP WON'T OVERRIDE
    dashed_line_1 = tikz.line(
        small_circle.west, 
        large_circle.west, 
        options="dashed, thin, black, KEEP_DASHED"
    )
    dashed_line_2 = tikz.line(
        small_circle.east, 
        large_circle.east, 
        options="dashed, thin, black, KEEP_DASHED"
    )
    
    return small_circle, large_circle

def create_wire_jump(center_point, radius=0.15, direction="up"):
    """Create a half-circle wire jump at the specified point"""
    if direction == "up":
        # Create an arc going upward (semicircle above the line)
        scope = Scope()
        scope.arc(center_point, 180, 0, radius=radius, options="ultra thick, black")
        tikz.draw(scope)
    else:
        # Create an arc going downward (semicircle below the line)
        scope = Scope()
        scope.arc(center_point, 0, 180, radius=radius, options="ultra thick, black")
        tikz.draw(scope)

# Input/Output tokens
input_tokens = ["Heute", " arbeitet", " er", "."]
target_tokens = ["Today", "he", "works", "."]

# Better horizontal spacing
dx = layer_block_width + 1.2

# Store references for connections
k_layers = []
p_layers = []
add_symbols_encoder = []
final_add_symbol = None

# Store references to sigmoid and output probability blocks
sigmoid_blocks = []
output_prob_blocks = []

# ENCODER
for i, token in enumerate(input_tokens[:3]):  # Process the first three tokens
    
    input_box = input_block(input_pos, token)
    
    sigmoid = sigmoid_block(Point(input_box.north.x, input_box.north.y))
    sigmoid_blocks.append(sigmoid)
    
    n_layer_block, n_output = vertical_layer_block(sigmoid.north, "N")
    
    # Create add symbol above the N layer
    add_pos = n_output + (0, 0.6)
    add_circle = add_symbol(add_pos, flip=False)
    add_symbols_encoder.append(add_circle)
    
    # Connect N layer output to add symbol WITH ARROW
    tikz.line(n_output, add_circle.south, options="ultra thick, ->, >=stealth, black")
    
    if i == 2:  # Last encoder timestep
        final_add_symbol = add_circle
    
    input_pos = input_pos + (dx, 0)

# Position K layers between consecutive add symbols
for i in range(len(add_symbols_encoder)):
    if i == 0:
        k_layer = horizontal_layer_block(Point(add_symbols_encoder[0].center.x - 2.0, add_symbols_encoder[0].center.y), "K")
        # No connection for the first K layer from previous add symbol

    else:
        prev_add = add_symbols_encoder[i-1]
        curr_add = add_symbols_encoder[i]
        k_x = (prev_add.center.x + curr_add.center.x) / 2
        k_y = (prev_add.center.y + curr_add.center.y) / 2
        k_layer = horizontal_layer_block(Point(k_x, k_y), "K")
        
        # Connect previous add symbol to this K layer
        tikz.line(prev_add.east, k_layer.west)
    
    k_layers.append(k_layer)
    
    # Connect K layer to current add symbol
    tikz.line(k_layer.east, add_symbols_encoder[i].west)
    
    # Add zoom effect to the second K layer (index 1)
    # if i == 1:
    #     create_zoom_effect(k_layer)

# ADD THE MISSING K LAYER AFTER THE FINAL ENCODER ADD SYMBOL
if final_add_symbol:
    # Position the final K layer to the right of the last encoder add symbol
    final_k_pos = Point(final_add_symbol.center.x + 1.6, final_add_symbol.center.y)
    final_k_layer = horizontal_layer_block(final_k_pos, "K")
    k_layers.append(final_k_layer)
    
    # Connect the final add symbol to this K layer
    tikz.line(final_add_symbol.east, final_k_layer.west)

# Calculate positions for labels and separator
encoder_center_x = add_symbols_encoder[1].center.x
encoder_decoder_gap = 2.5  # INCREASED GAP from 1.0 to 2.5
separator_x = encoder_center_x + (dx*1.5) + (encoder_decoder_gap / 2)

# Add vertical dashed separator line
separator_bottom = Point(separator_x-0.4, -2.5)
separator_top = Point(separator_x-0.4, 8.5)
tikz.line(separator_bottom, separator_top, options="dashed, gray!60, thick, KEEP_DASHED")

# Add section labels with reference formatting
tikz.node(Point(encoder_center_x, -2.5), text="\\textbf{Encoder}")

input_pos = Point(separator_x + encoder_decoder_gap, 0)

# Reset for decoder
add_symbols_decoder = []
dy = 0.6

# DECODER
for i, token in enumerate(input_tokens[:3]):
    target = target_tokens[i] if i < len(target_tokens) else None
    input_token = target_tokens[i-1] if i < len(target_tokens) and i > 0 else "$<$BOS$>$"

    input_box = input_block(input_pos, input_token)
    
    sigmoid = sigmoid_block(Point(input_box.north.x, input_box.north.y))
    sigmoid_blocks.append(sigmoid)
    
    l_layer_block, l_output = vertical_layer_block(sigmoid.north, "L")
    
    # Create add symbol above the L layer
    add_pos = l_output + (0, dy + 1)
    add_circle = add_symbol(add_pos, flip=False)
    add_symbols_decoder.append(add_circle)
    
    # Connect L layer output to add symbol WITH ARROW
    tikz.line(l_output, add_circle.south, options="ultra thick, ->, >=stealth, black")
    
    # Add M layer block above the add symbol
    m_layer_block, m_output = vertical_layer_block(add_circle.north, "M")
    
    # Add Group sum layer
    group_sum = group_sum_block(m_output)
    
    # Add Output Probabilities
    output_prob = output_probabilities(group_sum.north, target)
    output_prob_blocks.append(output_prob)
    
    input_pos = input_pos + (dx, 0)

# Calculate decoder center and add label
decoder_center_x = add_symbols_decoder[1].center.x
tikz.node(Point(decoder_center_x, -2.5), text="\\textbf{Decoder}")


# Position P layers between consecutive decoder add symbols
for i in range(len(add_symbols_decoder)):
    if i == 0:
        p_layer = horizontal_layer_block(Point(add_symbols_decoder[0].center.x - 2.0, add_symbols_decoder[0].center.y), "P")
        # No connection for the first P layer from previous add symbol
    else:
        prev_add = add_symbols_decoder[i-1]
        curr_add = add_symbols_decoder[i]
        p_x = (prev_add.center.x + curr_add.center.x) / 2
        p_y = (prev_add.center.y + curr_add.center.y) / 2
        p_layer = horizontal_layer_block(Point(p_x, p_y), "P")
        
        # Connect previous add symbol to this P layer
        tikz.line(prev_add.east, p_layer.west)
    
    p_layers.append(p_layer)
    
    # Connect P layer to current add symbol
    tikz.line(p_layer.east, add_symbols_decoder[i].west)

# Connect final encoder add symbol to all decoder add symbols WITH WIRE JUMPS
if final_add_symbol and add_symbols_decoder:
    connection_start = final_add_symbol.east
    rightmost_add_x = add_symbols_decoder[-1].center.x
    horizontal_y = connection_start.y
    
    # Calculate intersection points where horizontal line crosses vertical lines
    intersection_points = []
    for j, decoder_add in enumerate(add_symbols_decoder):
        intersection_x = decoder_add.center.x - 0.3
        intersection_points.append(Point(intersection_x, horizontal_y))
    
    # Draw the horizontal line in segments with wire jumps
    current_x = connection_start.x+1.75
    end_x = rightmost_add_x - 0.6
    
    # Draw line segments between intersections
    for i, intersection_point in enumerate(intersection_points):
        # Draw line segment before intersection
        if current_x < intersection_point.x +0.15 and i<2:
            tikz.line(
                Point(current_x, horizontal_y),
                Point(intersection_point.x +0.15, horizontal_y),
                options="ultra thick, black"
            )
        
            # Add wire jump (half-circle)
            create_wire_jump(intersection_point+(0.15,0.0), radius=0.15, direction="up")
            
            # Update current position after the jump
            current_x = intersection_point.x + 0.45
    
    # Draw final segment if needed
    if current_x < end_x:
        tikz.line(
            Point(current_x, horizontal_y),
            Point(end_x, horizontal_y),
            options="ultra thick, black"
        )
    
    # Vertical connections to each decoder add symbol (unchanged)
    for j, decoder_add in enumerate(add_symbols_decoder):
        add_x = decoder_add.center.x-0.3
        tikz.plot_coordinates(
            [Point(add_x-0.3, horizontal_y), decoder_add.south+(-0.05,0.05)],
            options="rounded corners, rounded corners=7pt, ultra thick, ->, >=stealth, black"
        )

# Reference-style styling - MODIFIED TO PRESERVE DASHED LINES
for obj in tikz.drawing_objects:
    if isinstance(obj, Rectangle):
        obj.options = f"{obj.options}, rounded corners=4pt, ultra thick"

    if isinstance(obj, Line):
        # Don't override options if they contain KEEP_DASHED marker or already have dashed
        if "KEEP_DASHED" in obj.options:
            # Remove the marker but keep the dashed formatting
            obj.options = obj.options.replace(", KEEP_DASHED", "")
        elif "dashed" in obj.options:
            # Keep existing dashed options
            pass
        elif len(obj.options) == 0:
            obj.options = "ultra thick, ->, >=stealth"
        elif "orange" not in obj.options and "gray" not in obj.options and "black" not in obj.options:
            obj.options = "ultra thick"

    if isinstance(obj, PlotCoordinates):
        if "rounded corners" not in obj.options:
            obj.options = (
                "rounded corners, rounded corners=7pt, ultra thick, ->, >=stealth"
            )

print(tikz.code())

\begin{center}
\begin{tikzpicture}
    \draw[fill=red!10, rounded corners=4pt, ultra thick] (-1.0, -0.625) rectangle (1.0, 0.625);
    \node[align=center] at (0.0, 0.0) { Input \\ Embedding };
    \node at (0.0, -1.325) { Heute };
    \draw[ultra thick, ->, >=stealth] (0.0, -1.075) to (0.0, -0.625);
    \draw[fill=green!10, rounded corners=4pt, ultra thick] (-1.0, 1.025) rectangle (1.0, 1.525);
    \node[align=center] at (0.0, 1.275) { Sigmoid };
    \draw[ultra thick, ->, >=stealth] (0.0, 0.625) to (0.0, 1.025);
    \draw[fill=ProcessBlue!10, rounded corners=4pt, ultra thick] (-0.4, 1.9249999999999998) rectangle (0.4, 2.7249999999999996);
    \node[align=center] at (0.0, 2.3249999999999997) { N };
    \draw[ultra thick, ->, >=stealth] (0.0, 1.525) to (0.0, 1.9249999999999998);
    \begin{scope}
	\draw[ultra thick] (0.0, 3.3249999999999997) circle (0.25cm);
	\draw[very thick] (0.0, 3.5749999999999997) to (0.0, 3.0749999999999997);
	\draw[very thick] (-0.25, 3.3249999999999997) to (0.25